In [1]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

In [2]:
PROJECT = Path(r"D:\master\Master_Drone_Detection")

MODEL_PATH = (
    PROJECT /
    "04_experiments" /
    "YOLO26s_P2_transfer_learning" /
    "weights" /
    "best.pt"
)

DATASET = PROJECT / "02_datasets" / "DUT_Anti_UAV"

VAL_IMAGES = DATASET / "images" / "val"
VAL_LABELS = DATASET / "labels" / "val"

model = YOLO(str(MODEL_PATH))

print("Model loaded")

Model loaded


In [3]:
def get_size(w, h, img_w, img_h):
    area = (w * img_w) * (h * img_h)

    if area < 1024:
        return "Tiny"
    elif area < 9216:
        return "Small"
    else:
        return "Medium"

In [4]:
def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0,x2-x1) * max(0,y2-y1)

    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])

    union = area1 + area2 - inter

    return inter / union if union else 0

In [ ]:
results = model.predict(
    source=str(VAL_IMAGES),
    imgsz=960,
    conf=0.25,
    save=False,
    verbose=False,
    stream=True
)

print("Prediction generator created")

Prediction generator created


In [10]:
stats = {
    "Tiny": {"TP":0,"FN":0,"GT":0},
    "Small": {"TP":0,"FN":0,"GT":0},
    "Medium": {"TP":0,"FN":0,"GT":0},
}


for result in tqdm(results):

    img_name = Path(result.path).name
    label_file = VAL_LABELS / img_name.replace(".jpg",".txt")

    img = cv2.imread(str(result.path))
    h,w = img.shape[:2]

    gt_boxes=[]

    if label_file.exists():

        labels = open(label_file).read().strip().splitlines()

        for line in labels:
            cls,x,y,bw,bh = map(float,line.split())

            x1=(x-bw/2)*w
            y1=(y-bh/2)*h
            x2=(x+bw/2)*w
            y2=(y+bh/2)*h

            size=get_size(bw,bh,w,h)

            gt_boxes.append(
                ([x1,y1,x2,y2],size)
            )

            stats[size]["GT"] +=1


    pred_boxes=[]

    for box in result.boxes.xyxy.cpu().numpy():
        pred_boxes.append(box.tolist())


    matched=set()


    for gt_box,size in gt_boxes:

        best_iou=0
        best_idx=-1

        for i,pred in enumerate(pred_boxes):

            if i in matched:
                continue

            iou=calculate_iou(gt_box,pred)

            if iou>best_iou:
                best_iou=iou
                best_idx=i


        if best_iou >=0.5:
            stats[size]["TP"]+=1
            matched.add(best_idx)

        else:
            stats[size]["FN"]+=1

2600it [02:39, 16.26it/s]


In [12]:
rows=[]

for size,data in stats.items():

    recall = data["TP"] / data["GT"] if data["GT"] else 0

    rows.append([
        size,
        data["TP"],
        data["FN"],
        data["GT"],
        recall
    ])


df=pd.DataFrame(
    rows,
    columns=[
        "Size",
        "TP",
        "FN",
        "Total_GT",
        "Recall"
    ]
)

df

,Size,TP,FN,Total_GT,Recall
0,Tiny,1285,117,1402,0.916548
1,Small,860,32,892,0.964126
2,Medium,318,9,327,0.972477
